# Time Series Imaging

Time series can be converted to **2D image representations** that expose
structural properties invisible in raw line plots. Once converted, standard
image analysis tools — CNNs, feature extractors, visual inspection — become
applicable to time series problems.

This notebook covers four imaging techniques planned for polars-ts:

| Technique | Module | Key idea |
|---|---|---|
| **Recurrence Plots** | `polars_ts.imaging.recurrence` | Visualise recurrence structure; extract RQA features |
| **Gramian Angular Fields** | `polars_ts.imaging.angular` | GASF/GADF — encode temporal correlations as angular differences |
| **Markov Transition Fields** | `polars_ts.imaging.transition` | Transition probabilities between quantile bins |
| **Spectrograms / Scalograms** | `polars_ts.imaging.spectral` | STFT and CWT frequency-time representations |

### Status

These modules are tracked in the [clustering roadmap (#100)](https://github.com/drumtorben/polars-ts/issues/100):

- #101 — Recurrence plots + RQA
- #102 — GAF + MTF
- #104 — Spectrograms + wavelets

Once implemented, the cells below will run end-to-end.

### References

- Eckmann et al. (1987). *Recurrence Plots of Dynamical Systems*. Europhysics Letters.
- Wang & Oates (2015). *Encoding Time Series as Images for Classification*. AAAI Workshop.
- Hatami et al. (2018). *Classification of TS by CNNs Applied to Recurrence Plots*.
- Li et al. (NeurIPS 2023). *Time Series as Images: ViT for Irregularly Sampled TS*.

In [ ]:
# Install polars-timeseries and hvplot if not already available
import importlib

if importlib.util.find_spec("polars_ts") is None:
    %pip install -q polars-timeseries[all]
if importlib.util.find_spec("hvplot") is None:
    %pip install -q hvplot

In [ ]:
import numpy as np
import polars as pl

# Load sample data — M4 hourly, first 10 series
df = (
    pl.scan_parquet("https://datasets-nixtla.s3.amazonaws.com/m4-hourly.parquet")
    .filter(pl.col("unique_id").str.strip_chars_start("H").cast(pl.Int64) <= 10)
    .collect()
)

# Z-normalize per series
df = df.with_columns(((pl.col("y") - pl.mean("y").over("unique_id")) / pl.std("y").over("unique_id")).alias("y"))

print(f"Shape: {df.shape}, Series: {df['unique_id'].n_unique()}")
df.head()

## 1. Recurrence Plots

A recurrence plot is a binary matrix **R** where `R[i,j] = 1` if the
distance between `x(i)` and `x(j)` is below a threshold `eps`.

- **Periodic** signals produce diagonal lines
- **Chaotic** signals produce complex textures
- **White noise** produces uniform scatter

Recurrence Quantification Analysis (RQA) extracts scalar features from
these plots: recurrence rate, determinism, entropy, laminarity.

In [ ]:
# Placeholder — will work once #101 is implemented
try:
    from polars_ts.imaging.recurrence import recurrence_plot, rqa_features

    # Generate recurrence plot for a single series
    series = df.filter(pl.col("unique_id") == "H1")["y"].to_numpy()
    rp = recurrence_plot(series, eps=0.3)

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(rp, cmap="binary", origin="lower")
    ax.set_title("Recurrence Plot — H1")
    ax.set_xlabel("Time")
    ax.set_ylabel("Time")
    plt.show()

    # RQA features
    features = rqa_features(rp)
    print("RQA features:", features)
except ImportError:
    print("polars_ts.imaging.recurrence not yet available.")
    print("Tracked in: https://github.com/drumtorben/polars-ts/issues/101")

## 2. Gramian Angular Fields & Markov Transition Fields

**GASF** (Gramian Angular Summation Field) and **GADF** (Gramian Angular
Difference Field) encode a time series as angular coordinates on the unit
circle, then compute pairwise sum/difference matrices.

**MTF** (Markov Transition Field) discretises values into quantile bins,
estimates a Markov transition matrix, and maps transitions back onto the
time axis.

In [ ]:
# Placeholder — will work once #102 is implemented
try:
    from polars_ts.imaging.angular import gadf, gasf
    from polars_ts.imaging.transition import mtf

    series = df.filter(pl.col("unique_id") == "H1")["y"].to_numpy()

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (img, title) in zip(
        axes, [(gasf(series), "GASF"), (gadf(series), "GADF"), (mtf(series), "MTF")], strict=False
    ):
        ax.imshow(img, cmap="viridis", origin="lower")
        ax.set_title(title)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("polars_ts.imaging.angular / transition not yet available.")
    print("Tracked in: https://github.com/drumtorben/polars-ts/issues/102")

## 3. Spectrograms & Wavelet Scalograms

**STFT spectrograms** show how the frequency content of a signal evolves
over time. **CWT scalograms** provide multi-resolution analysis using
continuous wavelet transforms.

Both are especially useful for non-stationary series where frequency
components appear, disappear, or shift over time.

In [ ]:
# Placeholder — will work once #104 is implemented
try:
    from polars_ts.imaging.spectral import scalogram, spectrogram

    series = df.filter(pl.col("unique_id") == "H1")["y"].to_numpy()

    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(spectrogram(series), aspect="auto", cmap="magma", origin="lower")
    axes[0].set_title("STFT Spectrogram")
    axes[1].imshow(scalogram(series), aspect="auto", cmap="magma", origin="lower")
    axes[1].set_title("CWT Scalogram")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("polars_ts.imaging.spectral not yet available.")
    print("Tracked in: https://github.com/drumtorben/polars-ts/issues/104")

## 4. Image-based clustering pipeline

Once imaging functions are available, the workflow is:

1. Generate images for each series (recurrence plots, GAF, or spectrograms)
2. Flatten images or extract CNN features
3. Cluster the resulting feature vectors
4. Compare with distance-based clusters from Notebook 07

In [ ]:
# Placeholder — will work once imaging modules are implemented
try:
    from sklearn.cluster import KMeans

    from polars_ts.imaging.recurrence import recurrence_plot

    # Generate recurrence plots for all series
    series_ids = df.select("unique_id").unique(maintain_order=True)["unique_id"].to_list()
    images = []
    for sid in series_ids:
        vals = df.filter(pl.col("unique_id") == sid)["y"].to_numpy()
        rp = recurrence_plot(vals, eps=0.3)
        images.append(rp.flatten())

    X = np.stack(images)
    km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X)

    labels = pl.DataFrame({"unique_id": series_ids, "cluster": km.labels_})
    print("Image-based cluster assignments:")
    print(labels)
except ImportError:
    print("Imaging modules not yet available.")
    print("See roadmap: https://github.com/drumtorben/polars-ts/issues/100")

## Summary

| Technique | Function | Output |
|---|---|---|
| Recurrence Plot | `recurrence_plot` | Binary NxN matrix |
| RQA Features | `rqa_features` | Dict of scalar features |
| GASF / GADF | `gasf`, `gadf` | NxN angular field matrix |
| MTF | `mtf` | NxN transition field matrix |
| Spectrogram | `spectrogram` | Frequency x time matrix |
| Scalogram | `scalogram` | Scale x time matrix |

### Workflow

1. **Image generation** — convert each series to a 2D array
2. **Feature extraction** — flatten, pool, or run through a CNN
3. **Downstream task** — cluster, classify, or detect anomalies

### Further reading

- Wang & Oates (2015). *Encoding Time Series as Images for Classification*. AAAI Workshop.
- Hatami et al. (2018). *Classification of TS by CNNs Applied to Recurrence Plots*.
- Eckmann et al. (1987). *Recurrence Plots of Dynamical Systems*. Europhysics Letters.
- Li et al. (NeurIPS 2023). *Time Series as Images: ViT for Irregularly Sampled TS*.